In [39]:
import pandas as pd

# Load matches
matches = pd.read_parquet(r"Datasets/SkillCorner Premier League 24-25 data/matches_clean.parquet")

# Create team lookup table
team_lookup = pd.concat([
    matches[["home_team_id","home_team_name"]].rename(
        columns={"home_team_id":"team_id","home_team_name":"team_name"}
    ),
    matches[["away_team_id","away_team_name"]].rename(
        columns={"away_team_id":"team_id","away_team_name":"team_name"}
    )
]).drop_duplicates()

print(team_lookup.head())

   team_id              team_name
0       31      Manchester United
1      752           Ipswich Town
2       37        West Ham United
3        3  Arsenal Football Club
4       41                Everton


In [41]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# PANDAS DISPLAY SETTINGS
# --------------------------------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)

# ==================================================
# 1. LOAD ALL MATCH EVENT FILES
# ==================================================
folder = Path(r"Datasets\SkillCorner Premier League 24-25 data\dynamic_events_pl_24\dynamic")
dfs = [pd.read_parquet(file) for file in folder.glob("*.parquet")]
events = pd.concat(dfs, ignore_index=True)

print("Total events loaded:", len(events))
print("Unique matches:", events["match_id"].nunique())

# ==================================================
# 2. FILTER PASSING OPTIONS
# ==================================================
passing_options = events[events["event_type"] == "passing_option"].copy()
print("\nPassing option rows:", len(passing_options))

# ==================================================
# 3. KEEP IMPORTANT COLUMNS & CLEAN
# ==================================================
passing_options = passing_options[
    [
        "match_id",
        "associated_player_possession_event_id",
        "player_id",
        "player_name",
        "passing_option_score",
        "xthreat",
        "player_targeted_xpass_completion",
        "targeted"
    ]
].dropna(subset=["passing_option_score"])

# Convert numeric columns
passing_options["xthreat"] = pd.to_numeric(passing_options["xthreat"], errors="coerce").fillna(0)
passing_options["player_targeted_xpass_completion"] = pd.to_numeric(
    passing_options["player_targeted_xpass_completion"], errors="coerce"
).fillna(1)

print("\nRows after cleaning:", len(passing_options))

# ==================================================
# 4. COUNT OPTIONS PER DECISION
# ==================================================
option_counts = (
    passing_options
    .groupby(["match_id", "associated_player_possession_event_id"])
    .size()
    .reset_index(name="n_options")
)
print("\nDecision moments:", len(option_counts))

# ==================================================
# 5. REMOVE TRIVIAL DECISIONS
# ==================================================
option_counts = option_counts[option_counts["n_options"] >= 2]
passing_options = passing_options.merge(
    option_counts[["match_id", "associated_player_possession_event_id"]],
    on=["match_id", "associated_player_possession_event_id"]
)
print("\nPassing options after filtering trivial decisions:", len(passing_options))

# ==================================================
# 6. PASS VALUE
# ==================================================
passing_options["pass_value"] = (
    passing_options["passing_option_score"]
    * passing_options["xthreat"]
    * passing_options["player_targeted_xpass_completion"]
)

# ==================================================
# 7. BEST OPTION PER DECISION
# ==================================================
best_option = (
    passing_options
    .groupby(["match_id", "associated_player_possession_event_id"])
    .agg(
        best_option_score=("passing_option_score", "max"),
        best_xThreat_available=("xthreat", "max"),
        best_pass_value=("pass_value", "max"),
        avg_option_quality=("passing_option_score", "mean"),
        avg_option_xThreat=("xthreat", "mean")
    )
    .reset_index()
)

# ==================================================
# 8. IDENTIFY CHOSEN PASSES
# ==================================================
chosen = passing_options[passing_options["targeted"] == True].copy()
chosen = chosen[
    [
        "match_id",
        "associated_player_possession_event_id",
        "player_id",
        "player_name",
        "passing_option_score",
        "xthreat",
        "pass_value"
    ]
].rename(columns={
    "passing_option_score": "chosen_option_score",
    "xthreat": "chosen_xThreat",
    "pass_value": "chosen_pass_value"
})

# ==================================================
# 9. MERGE DECISION DATA
# ==================================================
decisions = chosen.merge(
    best_option,
    on=["match_id", "associated_player_possession_event_id"]
)

# ==================================================
# 10. COMPUTE DECISION METRICS
# ==================================================
decisions["decision_quality"] = decisions["chosen_option_score"] / decisions["best_option_score"]
decisions["value_decision_quality"] = decisions["chosen_pass_value"] / decisions["best_pass_value"]
decisions["creativity"] = decisions["chosen_xThreat"] / decisions["best_xThreat_available"]
decisions["risk_taken"] = decisions["best_option_score"] - decisions["chosen_option_score"]
decisions["xThreat_gain"] = decisions["chosen_xThreat"] - decisions["best_xThreat_available"]

# ==================================================
# 11. AGGREGATE PLAYER METRICS
# ==================================================
players = (
    decisions
    .groupby(["player_id", "player_name"])
    .agg(
        passes=("decision_quality", "count"),
        decision_quality=("decision_quality", "mean"),
        value_decision_quality=("value_decision_quality", "mean"),
        creativity=("creativity", "mean"),
        avg_xThreat_created=("chosen_xThreat", "mean"),
        avg_xThreat_gain=("xThreat_gain", "mean"),
        total_xThreat_gain=("xThreat_gain", "sum"),
        avg_option_quality=("avg_option_quality", "mean"),
        risk_taken=("risk_taken", "mean")
    )
    .reset_index()
)

players = players[players["passes"] > 200]
print("Players with 200+ decisions:", len(players))

# ==================================================
# 12. LOAD PLAYER POSITION + MINUTES
# ==================================================
players_match = pd.read_parquet(r"Datasets/SkillCorner Premier League 24-25 data/players_match.parquet")
players_match["player_id"] = players_match["id"]
players_match["player_name"] = players_match["short_name"]
players_match["position"] = players_match["player_role_acronym"]
players_match["position_group"] = players_match["player_role_position_group"]

# Most common position
position_counts = (
    players_match
    .groupby(["player_id","player_name","team_id","position","position_group"])
    .size()
    .reset_index(name="matches_at_position")
)

player_main_position = (
    position_counts
    .sort_values("matches_at_position", ascending=False)
    .drop_duplicates("player_id")
)
player_main_position = player_main_position[
    ["player_id","team_id","position","position_group"]
].rename(columns={"position":"main_position"})

# Minutes played
minutes_df = (
    players_match
    .groupby("player_id")
    .agg(
        minutes_played_total=("playing_time_total_minutes_played","sum"),
        minutes_regular_time=("playing_time_total_minutes_played_regular_time","sum")
    )
    .reset_index()
)
minutes_df["per90_factor"] = minutes_df["minutes_played_total"] / 90

# Merge player info
player_info = player_main_position.merge(minutes_df, on="player_id", how="left")
players = players.merge(player_info, on="player_id", how="left")

# ==================================================
# 13. LOAD TEAM NAMES
# ==================================================
matches = pd.read_parquet(r"Datasets/SkillCorner Premier League 24-25 data/matches_clean.parquet")
team_lookup = pd.concat([
    matches[["home_team_id","home_team_name"]].rename(columns={"home_team_id":"team_id","home_team_name":"team_name"}),
    matches[["away_team_id","away_team_name"]].rename(columns={"away_team_id":"team_id","away_team_name":"team_name"})
]).drop_duplicates()

# Merge team names
players = players.merge(team_lookup, on="team_id", how="left")

# ==================================================
# 14. FINAL COLUMN ORDER
# ==================================================
columns_order = [
    "player_id", "player_name", "team_id", "team_name",
    "main_position", "position_group",
    "passes", "decision_quality", "value_decision_quality",
    "creativity", "avg_xThreat_created", "avg_xThreat_gain",
    "total_xThreat_gain", "avg_option_quality", "risk_taken",
    "minutes_played_total", "minutes_regular_time", "per90_factor"
]
players = players[columns_order]

print("\nFinal players table:")
print(players.head(10).to_string(index=False))

# ==================================================
# 15. BEST DECISION MAKERS EXAMPLE
# ==================================================
print("\nTop decision makers:")
print(players.sort_values("decision_quality", ascending=False).head(20).to_string(index=False))

Total events loaded: 1811078
Unique matches: 378

Passing option rows: 939059

Rows after cleaning: 939058

Decision moments: 330426

Passing options after filtering trivial decisions: 893754
Players with 200+ decisions: 374

Final players table:
 player_id  player_name  team_id               team_name main_position   position_group  passes  decision_quality  value_decision_quality  creativity  avg_xThreat_created  avg_xThreat_gain  total_xThreat_gain  avg_option_quality  risk_taken  minutes_played_total  minutes_regular_time  per90_factor
        82 A. Cresswell       37         West Ham United           SUB            Other     328          0.927248                0.376534    0.362124             0.001796         -0.005844             -1.9167            0.822113    0.064366                937.72                937.72     10.419111
       124  A. Doucouré       41                 Everton            AM         Midfield     831          0.904008                0.715670    0.710930      

In [ ]:
players